In [1]:
import os
import sys
import time
import psutil
from glob import glob
from functools import partial
from shutil import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

import pyscf
from pyscf import gto, scf, mcscf, cc
import ffsim

from qiskit import QuantumCircuit, QuantumRegister
from qiskit.primitives import StatevectorSampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from ansatzmap import get_zigzag_physical_layout
from DDLUCJ import DDLUCJ, GrabAmps

In [2]:


# Load metadata
moldf = pd.read_csv("molecules.csv")
activespacedf = pd.read_csv("active_spaces.csv")
energyDF = pd.read_csv("../../../classical/energies.csv", index_col=0)

BasisSets = ["STO-3G", "cc-pVDZ", "aug-cc-pVDZ"]

In [3]:
def run(pathxyz, name, basis, n_electrons, num_orbitals, L, k, shots):
    tag = f"{name}_LUCJ_L{L}_{basis}_{k}_StateVector_Shots{shots}"

    filecontents = f"""import os
import numpy as np
import pandas as pd

import pyscf
from pyscf import gto, scf, mcscf, cc
import ffsim

from qiskit import QuantumCircuit, QuantumRegister
from qiskit.primitives import StatevectorSampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from ansatzmap import get_zigzag_physical_layout
from DDLUCJ import DDLUCJ, GrabAmps

ampdict = GrabAmps("{name}", "{basis}")
t1, t2 = ampdict["{k}"]

initDDLUCJ = DDLUCJ(
    StructurePath="{pathxyz}",
    BasisSet="{basis}",
    NElec=int({n_electrons}),
    NOrb=int({num_orbitals}),
    Symmetry=False,  # avoid buggy orbsym path (same fix as block2/DMRG)
    injected=True,
    t1=t1,
    t2=t2,
    n_reps=int({L}),
    backend="statevector",
    optimization_level=3,
    shots=int({shots}),
    temp_dir="./",
    clean_temp_dir=True,
    n_jobs=1,
    num_batches=10,
    max_iterations=5,
    samples_per_batch=1000,
    verbose=True
)

total_energy, subspace_dim = initDDLUCJ(postprocess=True, usefulqrum=True)
energy_val = float(np.squeeze(total_energy))

EnergyPath = "../energies/{tag}.txt"
with open(EnergyPath, \'w\') as f:
    f.write(f"Basis Set: {basis}\\n")
    f.write(f"Molecule: {name}\\n")
    f.write(f"Method: LUCJ(L={L})/{k}\\n")
    f.write(f"Energy: {{energy_val}}\\n")
    f.write(f"Subspace Dimension: {{subspace_dim}}\\n")
"""

    with open(f"./postprocess/{tag}.py", "w") as f:
        f.write(filecontents)

    runfile = f"""#!/bin/bash
#SBATCH --time=0-8:00:00
#SBATCH -J {tag}
#SBATCH --account=rrg-jacobsen-ab
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=8
#SBATCH --mem-per-cpu=5GB
#SBATCH --error=job.e%J
#SBATCH --output=job.o%j

set -e

echo 'About to run python file'
module load python/3.11
module load StdEnv/2023
module load openmpi
module load symengine rust
module load hdf5
module load openblas

echo "TEMP DIR: $SLURM_TMPDIR"
source /home/gjones/scratch/tmp_env/bin/activate
# virtualenv --no-download $SLURM_TMPDIR/env
# source $SLURM_TMPDIR/env/bin/activate
# pip install --no-index --upgrade pip
# pip install --find-links=/home/gjones/projects/def-jacobsen/gjones/wheels/ fulqrum
# pip install -e /home/gjones/scratch/distributed_LUCJ/
# pip install openfermion

export LD_LIBRARY_PATH=$EBROOTOPENBLAS/lib:$LD_LIBRARY_PATH
export OMP_NUM_THREADS=$SLURM_CPUS_PER_TASK
export OPENBLAS_NUM_THREADS=$SLURM_CPUS_PER_TASK
export MKL_NUM_THREADS=$SLURM_CPUS_PER_TASK
export NUMEXPR_NUM_THREADS=$SLURM_CPUS_PER_TASK

echo "Running in directory: $(pwd)"
echo "{tag}"
python "{tag}.py"
echo "File run"
"""
    with open(f"./postprocess/{tag}.sh", "w") as f:
        f.write(runfile)

,molecule,formula,mol_filename,num_orbitals,n_electrons
0,ammonia,NH3,ammonia157.xyz,8,10
1,methane,CH4,methane50.xyz,9,10
2,ethylene,C2H4,ethylene42.xyz,14,16
3,ethane,C2H6,ethane28.xyz,16,18
4,water,H2O,water183.xyz,7,10
5,formaldehyde,CH20,formaldehyde138.xyz,12,16
6,methanol,CH3OH,methanol22.xyz,14,18
7,fluoroform,CHF3,GDB04_5.xyz,21,34
8,"buta-1,3-diene",C4H6,GDB04_53.xyz,26,30
9,but-1-yne,C4H6,GDB04_49.xyz,26,30


In [4]:
n_generated = 0
n_skipped = 0
shots = 10000
# for shots in np.logspace(4, 8,5).astype(int):
for row in tqdm(moldf.itertuples(), total=len(moldf), desc="Molecules"):
    moldict = row._asdict()
    name = moldict["molecule"]
    n_electrons = int(moldict["n_electrons"])
    num_orbitals = int(moldict["num_orbitals"])
    xyzname = moldict["mol_filename"]
    pathxyz = os.path.join("../../../../classical/structures/", xyzname)
    
    for basis in BasisSets:
        ampdict = GrabAmps(name, basis)
        for k in ampdict.keys():
            for L in range(1, 6):
                tag = f"{name}_LUCJ_L{L}_{basis}_{k}_StateVector_Shots{shots}"
                EnergyPath = f"./energies/{tag}.txt"

                if os.path.exists(EnergyPath):
                    n_skipped += 1
                    continue

                run(pathxyz, name, basis, n_electrons, num_orbitals, L, k, shots)
                n_generated += 1

print(f"Generated: {n_generated}  |  Skipped (already complete): {n_skipped}")

Molecules:   0%|          | 0/12 [00:00<?, ?it/s]

Generated: 900  |  Skipped (already complete): 180


In [5]:
len(glob("./postprocess/*_StateVector_*.sh"))

900

In [6]:
# Submit generated jobs to Nibi (uncomment when ready to fire)
# import subprocess
# for shfile in sorted(glob("./postprocess/*_StateVector_*.sh")):
#     tag = os.path.basename(shfile)[:-3]
#     result = subprocess.run(
#         ["sbatch", os.path.basename(shfile)],
#         cwd="./postprocess",
#         capture_output=True, text=True
#     )
#     jobid = result.stdout.strip().split()[-1]
#     with open(f"./jobids/{tag}.txt", "w") as f:
#         f.write(f"{tag}\n{jobid}\n")
#     print(tag, jobid)

In [7]:
refdf = pd.read_csv("../../../classical/energies.csv", index_col=0).reset_index(drop=True)

In [8]:
refdf.loc[(refdf["Name"] == "ammonia") & (refdf["Basis Set"] == "STO-3G")]

,Method,Energy,Name,Basis Set
3,HF,-55.454318,ammonia,STO-3G
4,CASCI,-55.519929,ammonia,STO-3G
5,SCI,-55.519894,ammonia,STO-3G
95,DMRG-CASCI,-55.519929,ammonia,STO-3G


In [10]:
glob("energies/*StateVector*")

['energies/ammonia_LUCJ_L1_cc-pVDZ_ML_exact_StateVector_Shots10000.txt',
 'energies/methane_LUCJ_L1_STO-3G_zeroes_StateVector_Shots10000.txt',
 'energies/ammonia_LUCJ_L1_aug-cc-pVDZ_MP2_StateVector_Shots10000.txt',
 'energies/methane_LUCJ_L3_STO-3G_MP2_StateVector_Shots10000.txt',
 'energies/methane_LUCJ_L1_STO-3G_random_StateVector_Shots10000.txt',
 'energies/ammonia_LUCJ_L4_aug-cc-pVDZ_CCSD_StateVector_Shots10000.txt',
 'energies/methane_LUCJ_L2_STO-3G_MP2_StateVector_Shots10000.txt',
 'energies/methane_LUCJ_L2_aug-cc-pVDZ_ML_exact_StateVector_Shots10000.txt',
 'energies/methane_LUCJ_L1_STO-3G_MP2_StateVector_Shots10000.txt',
 'energies/ammonia_LUCJ_L4_cc-pVDZ_MP2_StateVector_Shots10000.txt',
 'energies/ammonia_LUCJ_L2_STO-3G_ML_StateVector_Shots10000.txt',
 'energies/methane_LUCJ_L5_aug-cc-pVDZ_MP2_StateVector_Shots10000.txt',
 'energies/ammonia_LUCJ_L5_STO-3G_ML_StateVector_Shots10000.txt',
 'energies/ammonia_LUCJ_L1_aug-cc-pVDZ_CCSD_StateVector_Shots10000.txt',
 'energies/ammonia_